In [2]:
import torch
from torch import nn
import torch.optim as optim

%matplotlib inline
torch.set_printoptions(sci_mode=False,precision=4)
torch.manual_seed(1234)
torch.random.manual_seed(1234)
generator = torch.Generator().manual_seed(1234)

In [3]:
context_length = 7
batch_size = 1
embd_dim = 2
iteration_count = 200
head_size = 4

In [4]:
with open('../data/numeric_data.txt', 'r', encoding = 'utf-8') as file:
    records = file.read()
print(records)

one hot


In [5]:
data = ''.join(records)
data = '_' * context_length + data
unique_text = sorted(list(set(data)))
vocab_size = len(unique_text)
print(vocab_size)
print(data)

7
_______one hot


In [6]:
stoi = {text: index for index, text in enumerate(unique_text)}
itos = {index: text for index, text in enumerate(unique_text)}

encoder = lambda char_array: [stoi[char] for char in ''.join(char_array)]
decoder = lambda num_array: [itos[num] for num in num_array]

print(stoi)

{' ': 0, '_': 1, 'e': 2, 'h': 3, 'n': 4, 'o': 5, 't': 6}


In [7]:
x_numpy = []
y_numpy = []
temp_data = data

for t in range(len(temp_data) - context_length - 1):
    x_numpy.append(encoder(temp_data[t : t + context_length]))
    y_numpy.append(stoi[temp_data[t + context_length]])

x = torch.tensor(x_numpy, dtype=torch.long)
y = torch.tensor(y_numpy, dtype=torch.long)

N = x.shape[0]
train_ratio = 0.99
val_ratio = 0.01

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:], y[train_end:]

In [8]:
criterion = nn.CrossEntropyLoss()
embedding = nn.Embedding(vocab_size, embd_dim)
logit_layer = nn.Linear(context_length * embd_dim, vocab_size)

embedding = nn.Embedding(vocab_size, embd_dim) # Assuming vocab_size is defined
pos_embedding = nn.Embedding(context_length, embd_dim)
key = nn.Linear(embd_dim, head_size, bias=False)
query = nn.Linear(embd_dim, head_size, bias=False)
value = nn.Linear(embd_dim, head_size, bias=False)
logit_layer = nn.Linear(head_size * context_length, vocab_size) # Adjusted input size

In [12]:
# --- Optimizer ---
# Note: embedding and logit_layer should be in .train() mode for training
optimizer = optim.Adam([
    {'params': list(embedding.parameters()), 'lr': 0.001},
    {'params': list(pos_embedding.parameters()), 'lr': 0.001}, # Add this
    {'params': list(key.parameters())},
    {'params': list(query.parameters())},
    {'params': list(value.parameters()), 'lr': 0.001},
    {'params': list(logit_layer.parameters()), 'lr': 0.001}
], lr=0.001)
iteration_count = 1
tril = torch.tril(torch.ones(context_length, context_length))
for epoch in range(1000):
    # 1. Data Fetching
    idx = torch.randint(0, len(x_train), (batch_size,), generator=generator)
    rand_x = x_train[idx] # Shape: (batch, context_length)
    rand_y = y_train[idx]
    # print('rand_y : ', rand_y.shape)

    optimizer.zero_grad(set_to_none=True)

    # 2. Embedding
    T = x.size(1)
    tok_emb = embedding(rand_x)
    print(tok_emb)
    pos = torch.arange(T, device=rand_x.device)
    pos_emb = pos_embedding(pos)
    x = tok_emb + pos_emb

    # 3. Self-Attention Mechanism
    k = key(x)   # (B, T, head_size)
    q = query(x) # (B, T, head_size)

    # print(key.weight.cpu().detach().numpy()[0])

    # print('x.size() : ', x.size())
    # print('q.size() : ', q.size())
    # print('k.size() : ', k.size())

    # Calculate attention scores (affinities)
    wei = q @ k.transpose(-2, -1) * (head_size**-0.5) # Scaled dot-product
    # wei = q @ k.transpose(-1, 0) * (head_size**-0.5) # Scaled dot-product
    # print('wei (1) : ', wei)

    wei = wei.masked_fill(tril == 0, float('-inf'))

    # print('wei (2) : ', wei)

    wei = torch.nn.functional.softmax(wei, dim=-1)

    # print('wei (3) : ', wei)

    # print('wei.size() : ', wei.size())

    # Apply weights to values
    v = value(x) # (B, T, head_size)
    out = wei @ v # (B, T, head_size)

    # print('out.size() : ', out.size())
    # out = wei @ x # (B, T, head_size)

    # 4. Preparation for Logit Layer
    # Flatten the sequence: (B, T, H) -> (B, T*H)
    out_flattened = out.view(batch_size, -1)

    # print('out_flattened.shape : ', out_flattened.shape)

    logits = logit_layer(out_flattened)
    # print('logits.shape : ', logits.shape)
    loss = criterion(logits, rand_y)

    # 5. Backprop
    if epoch % 100 == 0:
        print(loss.item())

    loss.backward()
    optimizer.step()

KeyboardInterrupt: 

In [ ]:
#Test
torch.set_printoptions(sci_mode=False, precision=2, threshold=float('inf'))
print(stoi)
logging = True
final_text = ("___________on")
for _ in range(3):
        T = x.size(1)
        x = torch.tensor(encoder(final_text[-context_length:]))
        x = torch.unsqueeze(x, 0)
        # print(x.shape)
        tok_emb = embedding(x) # (B, T, Embd_Size)
        pos = torch.arange(T, device=x.device)
        pos_emb = pos_embedding(pos)
        x = tok_emb + pos_emb


        # 2. Self-Attention (Must match training logic exactly)
        k = key(x)   # (B, T, head_size)

        # print('k 1 : ', k)
        q = query(x) # (B, T, head_size)

        # print('q 1 : ', q)
        # Calculate attention scores
        # Note: 'tril' should be accessible globally or passed in
        wei = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5)
        # Use a local mask in case T varies
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        wei = wei.masked_fill(mask == 0, float('-inf'))

        # print('Wei 1 : ', wei)

        wei = torch.nn.functional.softmax(wei, dim=-1)

        print('Wei 2 : ', wei)

        # Apply weights to values
        v = value(x)

        # print(v)
        out = wei @ v # (B, T, head_size)

        # print('out: ', out)

        # print(out)
        # 3. Flatten and get Logits
        # Ensure this view matches the shape used in training
        out_flattened = out.reshape(out.size(0), -1)
        logits = logit_layer(out_flattened)
        final = torch.max(logits, 1)
        y = decoder([final[1].item()])
        final_text+=y[0]

# print(logit_layer.weight.size())
print(final_text)
#pox o'' yo

In [ ]:
# print(x.shape)
# first = embedding.weight[:, 5].detach().float().cpu().numpy()
# second = embedding.weight[:, 4].detach().float().cpu().numpy()
# for i in range(len(first)):
#     word = itos[i]
#     if word in ['B','e','f','o', 'r', 'e', ' ','w','e']:
#         plt.scatter(first[i], second[i], s=500)
#         plt.text(first[i], second[i], word, fontsize=10, ha='center', va='center')

In [ ]:
# # --- Optimizer ---
# # Note: embedding and logit_layer should be in .train() mode for training
# optimizer = optim.Adam([
#     {'params': list(embedding.parameters()), 'lr': 0.001},
#     {'params': list(pos_embedding.parameters()), 'lr': 0.001}, # Add this
#     {'params': list(key.parameters())},
#     {'params': list(query.parameters())},
#     {'params': list(value.parameters()), 'lr': 0.001},
#     {'params': list(logit_layer.parameters()), 'lr': 0.001}
# ], lr=0.01)
#
# tril = torch.tril(torch.ones(context_length, context_length))
# for epoch in range(iteration_count * 4):
#     # 1. Data Fetching
#     idx = torch.randint(0, len(x_train), (batch_size,), generator=generator)
#     rand_x = x_train[idx] # Shape: (batch, context_length)
#     rand_y = y_train[idx]
#     # print('rand_y : ', rand_y.shape)
#
#     optimizer.zero_grad(set_to_none=True)
#
#     # 2. Embedding
#     T = x.size(1)
#     tok_emb = embedding(rand_x)
#     pos = torch.arange(T, device=rand_x.device)
#     pos_emb = pos_embedding(pos)
#     x = tok_emb + pos_emb
#
#     # 3. Self-Attention Mechanism
#     k = key(x)   # (B, T, head_size)
#     q = query(x) # (B, T, head_size)
#
#     # print(key.weight.cpu().detach().numpy()[0])
#
#     # print('x.size() : ', x.size())
#     # print('q.size() : ', q.size())
#     # print('k.size() : ', k.size())
#
#     # Calculate attention scores (affinities)
#     wei = q @ k.transpose(-2, -1) * (head_size**-0.5) # Scaled dot-product
#     # wei = q @ k.transpose(-1, 0) * (head_size**-0.5) # Scaled dot-product
#     # print('wei (1) : ', wei)
#
#     wei = wei.masked_fill(tril == 0, float('-inf'))
#
#     # print('wei (2) : ', wei)
#
#     wei = torch.nn.functional.softmax(wei, dim=-1)
#
#     # print('wei (3) : ', wei)
#
#     # print('wei.size() : ', wei.size())
#
#     # Apply weights to values
#     v = value(x) # (B, T, head_size)
#     out = wei @ v # (B, T, head_size)
#
#     # print('out.size() : ', out.size())
#     # out = wei @ x # (B, T, head_size)
#
#     # 4. Preparation for Logit Layer
#     # Flatten the sequence: (B, T, H) -> (B, T*H)
#     out_flattened = out.view(batch_size, -1)
#
#     # print('out_flattened.shape : ', out_flattened.shape)
#
#     logits = logit_layer(out_flattened)
#     # print('logits.shape : ', logits.shape)
#     loss = criterion(logits, rand_y)
#
#     # 5. Backprop
#     if epoch % 100 == 0:
#         print(loss.item())
#
#     loss.backward()
#     optimizer.step()